# 02 — End-to-End Pipeline Demo

This notebook demonstrates the complete pipeline:
1. Parse a job description
2. Score all resumes against it
3. Rank candidates
4. Analyse skill gaps
5. Generate explanations
6. Visualise results

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from src.skill_extractor import SkillExtractor
from src.jd_parser import JDParser
from src.scorer import ResumeScorer
from src.ranker import rank_candidates
from src.skill_gap import analyse_skill_gap, compare_candidates
from src.explain import explain_candidate, explain_ranking, explain_score_breakdown

# Load data
with open('../data/sample_resumes.json') as f:
    resumes = json.load(f)
with open('../data/sample_jds.json') as f:
    jds = json.load(f)

print(f'✓ Loaded {len(resumes)} resumes and {len(jds)} JDs')

## Step 1: Initialise the Pipeline

We share one `SkillExtractor` instance across all modules to avoid
reloading the spaCy model and taxonomy multiple times.

In [ ]:
# Shared extractor (loads spaCy + taxonomy once)
extractor = SkillExtractor()
parser = JDParser(skill_extractor=extractor)
scorer = ResumeScorer(skill_extractor=extractor)

print('✓ Pipeline initialised')

## Step 2: Parse the Job Description

In [ ]:
# Use the Senior Data Scientist JD
jd = jds[0]
parsed_jd = parser.parse(jd['text'], jd['title'])

print(f'Title: {parsed_jd.title}')
print(f'Experience: {parsed_jd.experience_years}+ years')
print(f'Seniority: {parsed_jd.seniority_level}')
print(f'\nRequired Skills ({len(parsed_jd.required_skills)}):')
for s in sorted(parsed_jd.required_skills):
    print(f'  ★ {s}')
print(f'\nPreferred Skills ({len(parsed_jd.preferred_skills)}):')
for s in sorted(parsed_jd.preferred_skills):
    print(f'  ☆ {s}')

## Step 3: Score & Rank All Candidates

In [ ]:
# Score all resumes against the JD
scored = scorer.score_batch(resumes, parsed_jd)

# Rank them
ranked = rank_candidates(scored)

# Display as a clean table
rows = []
for r in ranked:
    rows.append({
        'Rank': r['rank'],
        'Candidate': r['candidate'],
        'Score': r['final_score'],
        'Tier': r['tier'],
        'Text Sim': r['score_breakdown']['text_similarity'],
        'Req Skills': r['score_breakdown']['required_skills_score'],
        'Pref Skills': r['score_breakdown']['preferred_skills_score'],
        'Matched': len(r['matched_skills']),
        'Missing Req': len(r['missing_required_skills']),
    })

df = pd.DataFrame(rows)
df

## Step 4: Plain-English Explanations

In [ ]:
# Overall ranking summary
print(explain_ranking(ranked))
print()

In [ ]:
# Per-candidate explanations
for r in ranked[:5]:  # Top 5
    print(f"--- #{r['rank']} {r['candidate']} ---")
    print(explain_candidate(r))
    print(explain_score_breakdown(r))
    print()

## Step 5: Skill Gap Analysis

In [ ]:
# Detailed gap report for top 3
gap_reports = compare_candidates(
    ranked[:3],
    parsed_jd.required_skills,
    parsed_jd.preferred_skills,
    extractor
)

for gap in gap_reports:
    print(f"=== {gap['candidate']} (Score: {gap['final_score']}) ===")
    print(f"  Gap Severity: {gap['gap_severity']}")
    print(f"  Required: {gap['required_match_pct']}% matched")
    print(f"  Matched Required: {gap['matched_required']}")
    print(f"  Missing Required: {gap['missing_required']}")
    print(f"  Bonus Skills: {gap['bonus_skills']}")
    print()

## Step 6: Visualisations

In [ ]:
# Bar chart: Final scores
fig = px.bar(
    df.sort_values('Score', ascending=True),
    x='Score', y='Candidate',
    orientation='h',
    color='Tier',
    color_discrete_map={
        'Strong Match': '#22c55e',
        'Moderate Match': '#f59e0b',
        'Weak Match': '#ef4444'
    },
    title='Candidate Ranking — Final Scores',
    labels={'Score': 'Score (0-100)', 'Candidate': ''},
)
fig.update_layout(showlegend=True, height=400)
fig.show()

In [ ]:
# Grouped bar chart: Score breakdown comparison
fig = go.Figure()
top5 = df.head(5)

fig.add_trace(go.Bar(name='Text Similarity', x=top5['Candidate'], y=top5['Text Sim'], marker_color='#6366f1'))
fig.add_trace(go.Bar(name='Required Skills', x=top5['Candidate'], y=top5['Req Skills'], marker_color='#22c55e'))
fig.add_trace(go.Bar(name='Preferred Skills', x=top5['Candidate'], y=top5['Pref Skills'], marker_color='#f59e0b'))

fig.update_layout(
    barmode='group',
    title='Score Breakdown — Top 5 Candidates',
    yaxis_title='Score (%)',
    height=400,
)
fig.show()

In [ ]:
# Radar chart: Top 3 candidate comparison
categories = ['Text Similarity', 'Required Skills', 'Preferred Skills']

fig = go.Figure()
colors = ['#6366f1', '#22c55e', '#f59e0b']

for i, r in enumerate(ranked[:3]):
    values = [
        r['score_breakdown']['text_similarity'],
        r['score_breakdown']['required_skills_score'],
        r['score_breakdown']['preferred_skills_score'],
    ]
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],  # Close the polygon
        theta=categories + [categories[0]],
        fill='toself',
        name=r['candidate'],
        opacity=0.6,
        line_color=colors[i],
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title='Radar Comparison — Top 3 Candidates',
    height=500,
)
fig.show()

## Step 7: Export Results as JSON

In [ ]:
# Build the final output format
output = []
for r in ranked:
    output.append({
        'candidate': r['candidate'],
        'rank': r['rank'],
        'tier': r['tier'],
        'final_score': r['final_score'],
        'score_breakdown': r['score_breakdown'],
        'matched_skills': r['matched_skills'],
        'missing_required_skills': r['missing_required_skills'],
        'missing_preferred_skills': r['missing_preferred_skills'],
        'explanation': explain_candidate(r),
    })

# Pretty print the top result
print(json.dumps(output[0], indent=2))

## Bonus: Try with the Full Stack Web Developer JD

In [ ]:
# Same pipeline, different JD — shows how rankings change
jd2 = jds[1]
parsed_jd2 = parser.parse(jd2['text'], jd2['title'])
scored2 = scorer.score_batch(resumes, parsed_jd2)
ranked2 = rank_candidates(scored2)

print(f'=== Rankings for: {jd2["title"]} ===\n')
for r in ranked2:
    print(f'#{r["rank"]} {r["candidate"]}: {r["final_score"]}/100 ({r["tier"]})')